# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`


In [6]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


In [12]:
import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [13]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


↪ Ya existe: attention_paper.pdf
↪ Ya existe: rag_paper.pdf
✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres

Inicio del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


Attention → 55 chunks
RAG       → 95 chunks
Total     → 150 chunks

Ejemplo de metadatos: {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_index': 0}


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [15]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
)
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ 150 chunks indexados en Qdrant
✅ Dimensión de embeddings: 2048


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [16]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


Pregunta: What is the Transformer architecture?
Respuesta: The Transformer is a network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. It uses stacked self-attention and point-wise fully connected layers for both encoder and decoder. The encoder consists of 6 identical layers, each with a multi-head self-attention sub-layer and a position-wise feed-forward sub-layer, with residual connections and layer normalization (output dimension d=512). This design enables greater parallelization and achieves state-of-the-art results in machine translation.  
**Source:** *Attention Is All You Need*


In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = [
    (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
    (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf"),
]

def build_store(chunk_size: int, chunk_overlap: int):
    """Chunking + embeddings + Qdrant en memoria. Devuelve (store, n_chunks)."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    docs = []
    for text, title, source in CORPUS:
        for i, chunk in enumerate(splitter.split_text(text)):
            if len(chunk.strip()) >= 30:
                docs.append(Document(page_content=chunk,
                                     metadata={"title": title, "source": source, "chunk_index": i}))
    name = f"exp_{chunk_size}_{chunk_overlap}"
    client = QdrantClient(":memory:")
    client.create_collection(collection_name=name,
                             vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    store.add_documents(docs)
    return store, len(docs)


def build_chain(store, k: int = 5, threshold: float = 0.25):
    """Reutiliza un índice ya construido y cambia solo la recuperación."""
    retr = store.as_retriever(search_type="similarity_score_threshold",
                              search_kwargs={"k": k, "score_threshold": threshold})
    chain = ({"context": retr | format_docs, "question": RunnablePassthrough()}
             | prompt | llm | StrOutputParser())
    return chain, retr

In [27]:
preguntas = [q1, q2, q3, q4, q5]  # usa las variables de tus 5 preguntas del quiz

for cs in [400, 900, 1500]:
    overlap = round(cs * 0.13)
    store, n = build_store(cs, overlap)
    chain, retr = build_chain(store, k=5, threshold=0.25)
    aciertos = 0
    print(f"\n--- chunk_size={cs}, overlap={overlap}, chunks={n} ---")
    for q in preguntas:
        try:
            resp = chain.invoke(q)
            print(f"Q: {q[:50]}...\nA: {resp[:150]}...\n")
        except Exception as e:
            print(f"Error: {e}")


--- chunk_size=400, overlap=52, chunks=326 ---
Q: What BLEU score did the Transformer (big model) ac...
A: The Transformer (big model) achieved a BLEU score of **28.4** on the WMT 2014 English-to-German translation task.  
*Source: "Attention Is All You Nee...

Q: How does RAG combine parametric and non-parametric...
A: RAG combines parametric and non-parametric memory by endowing a pre-trained seq2seq transformer (parametric memory) with a dense vector index of Wikip...

Q: What is the purpose of the scaling factor sqrt(dk)...
A: The scaling factor \(1/\sqrt{d_k}\) is used to counteract the problem that for large values of \(d_k\), the dot products grow large in magnitude, push...

Q: What are the two RAG model variants proposed in th...
A: The paper proposes two RAG variants:

1. **RAG-Sequence** – uses the **same retrieved document(s)** for the entire generated sequence (conditions on t...

Q: How many attention heads does the base Transformer...
A: The base Transformer model uses 

In [30]:
import time

def invoke_con_reintento(chain, pregunta, intentos=4, espera=15):
    for i in range(intentos):
        try:
            return chain.invoke(pregunta)
        except Exception as e:
            if "503" in str(e) and i < intentos - 1:
                print(f"  (sobrecargado, reintentando en {espera}s...)")
                time.sleep(espera)
            else:
                return f"[ERROR: {e}]"

# Experimento 2 — k
for k in [3, 5, 8]:
    chain, retr = build_chain(store_900, k=k, threshold=0.25)
    print(f"\n--- k={k} ---")
    for q in preguntas:
        resp = invoke_con_reintento(chain, q)
        print(f"Q: {q[:50]}...\nA: {resp[:150]}...\n")
        time.sleep(2)  # pequeña pausa entre preguntas para no saturar más


--- k=3 ---
Q: What BLEU score did the Transformer (big model) ac...
A: [ERROR: HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=60)]...

Q: How does RAG combine parametric and non-parametric...
A: RAG combines a pre-trained seq2seq model (parametric memory) with a dense vector index of Wikipedia accessed via a pre-trained neural retriever (non-p...

Q: What is the purpose of the scaling factor sqrt(dk)...
A: The scaling factor \(1/\sqrt{d_k}\) in Scaled Dot-Product Attention prevents the dot products from growing too large in magnitude when \(d_k\) is larg...

Q: What are the two RAG model variants proposed in th...
A: The paper "Retrieval-Augmented Generation" proposes two RAG variants:

1. **RAG-Sequence**: Uses the **same retrieved document** for the entire genera...

Q: How many attention heads does the base Transformer...
A: The base Transformer model uses **8 attention heads** (h = 8), with key and value dimensions of **64 each per h

In [31]:
for th in [0.15, 0.25, 0.40]:
    chain, retr = build_chain(store_900, k=5, threshold=th)
    print(f"\n--- threshold={th} ---")
    for q in preguntas:
        resp = invoke_con_reintento(chain, q)
        print(f"Q: {q[:50]}...\nA: {resp[:150]}...\n")
        time.sleep(2)


--- threshold=0.15 ---
Q: What BLEU score did the Transformer (big model) ac...
A: The Transformer (big model) achieved a BLEU score of **28.4** on the WMT 2014 English-to-German translation task.

**Source:** *Attention Is All You N...

  (sobrecargado, reintentando en 15s...)
Q: How does RAG combine parametric and non-parametric...
A: RAG combines parametric and non-parametric memory by using a pre-trained seq2seq transformer (BART) as parametric memory and a dense vector index of W...

Q: What is the purpose of the scaling factor sqrt(dk)...
A: The scaling factor \(1/\sqrt{d_k}\) prevents the dot products from growing too large in magnitude when \(d_k\) is large, which would push the softmax ...

Q: What are the two RAG model variants proposed in th...
A: The paper proposes two RAG model variants:

1. **RAG-Sequence** – treats the retrieved document as a single latent variable for the entire generated s...

Q: How many attention heads does the base Transformer...
A: The base Transf

---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [17]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: The Transformer (big model) achieved a BLEU score of **28.4** on the WMT 2014 English-to-German translation task.

**Source:** *Attention Is All You Need* (attention_paper.pdf) - "On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big) in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0 BLEU, establishing a new state-of-the-art BLEU score of 28.4."


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [19]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

Q: How does RAG combine parametric and non-parametric memory?
A: RAG combines parametric and non-parametric memory by using a pre-trained seq2seq transformer (BART) as parametric memory and a dense vector index of Wikipedia accessed via a pre-trained neural retriever (DPR) as non-parametric memory. These components are integrated in a probabilistic model trained end-to-end: the retriever provides latent documents conditioned on the input, and the generator conditions on both the input and retrieved documents to produce the output. The model marginalizes over retrieved documents using a top-K approximation, either assuming the same document for the entire output (per-output basis) or allowing different documents per token (per-token basis). The non-parametric component guides generation by drawing out specific knowledge stored in the parametric memory.

[Retrieval-Augmented Generation | rag_paper.pdf]


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [20]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor \(1/\sqrt{d_k}\) in Scaled Dot-Product Attention prevents the dot products from growing too large in magnitude when \(d_k\) is large. Without scaling, large dot products push the softmax function into regions with extremely small gradients, hindering training. The scaling counteracts this effect by keeping the variance of the dot products stable (since the dot product of independent random variables with variance 1 has variance \(d_k\)).

**Source:** *Attention Is All You Need* (attention_paper.pdf)


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [21]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The paper proposes two RAG model variants:

1. **RAG-Sequence**: Uses the same retrieved document to generate the entire output sequence. It treats the retrieved document as a single latent variable marginalized over the whole sequence (per-output basis).

2. **RAG-Token**: Can predict each target token based on a different retrieved document. It marginalizes latent documents on a per-token basis, allowing different documents to be responsible for different tokens.

The key difference is in how they marginalize over retrieved documents: RAG-Sequence assumes one document generates all tokens, while RAG-Token allows different documents per token.

**Paper title**: Retrieval-Augmented Generation


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [24]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model uses **8 attention heads** (h = 8), with **key and value dimensions of 64 each per head** (d_k = d_v = d_model/h = 64).

**Source:** *Attention Is All You Need* (attention_paper.pdf)


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [25]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation. The Attention paper proposes the Transformer architecture. What seq2seq model does RAG use as its generator component, and is it based on the Transformer?

A: Based on the RAG paper excerpts, RAG uses **BART** as its seq2seq generator component. The paper explicitly states: "the seq2seq model (BART[32]) then conditions on these latent documents together with the input to generate the output" and refers to "a pre-trained seq2seq transformer" with BART as the implementation.

Since BART is a seq2seq transformer model (the paper groups it with T5 as "Like T5[51] or BART"), it **is based on the Transformer architecture** proposed in the Attention paper.

**Supporting paper:** Retrieval-Augmented Generation

Fuentes recuperadas:
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — r

<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

In [32]:
preguntas_trampa = [
    "What GPT-4 benchmark results are reported in the Attention paper?",
    "How does RAG perform compared to GPT-4 on open-domain QA?",
    "What is the BLEU score of the Transformer model on the WMT 2020 translation task?",
]

print("=== B3: Preguntas trampa ===")
for q in preguntas_trampa:
    resp = invoke_con_reintento(rag_chain, q)
    print(f"\nQ: {q}\nA: {resp}\n")
    time.sleep(2)

=== B3: Preguntas trampa ===

Q: What GPT-4 benchmark results are reported in the Attention paper?
A: The provided excerpts from "Attention Is All You Need" do not contain any GPT-4 benchmark results. The paper was published in 2017, while GPT-4 was released in 2023, so it could not possibly report GPT-4 results. The benchmarks mentioned in the excerpts are BLEU scores on WMT 2014 English-to-German (28.4 BLEU) and English-to-French (41.8 BLEU) translation tasks, as well as comparisons with the Berkeley Parser and Recurrent Neural Network Grammar on constituency parsing.


Q: How does RAG perform compared to GPT-4 on open-domain QA?
A: The provided excerpts from the "Retrieval-Augmented Generation" paper do not contain any comparison between RAG and GPT-4 on open-domain QA. The paper compares RAG to models such as REALM, T5+SSM, DPR, BART, and other extractive/closed-book QA approaches, but GPT-4 is not mentioned.


Q: What is the BLEU score of the Transformer model on the WMT 2020 tran

In [33]:
preguntas_es = [
    "¿Qué puntuación BLEU logró el Transformer (modelo grande) en WMT 2014 inglés-alemán?",
    "¿Cuáles son las dos variantes del modelo RAG propuestas en el paper?",
    "¿Cuántas cabezas de atención usa el Transformer base?",
]
preguntas_en_equiv = [
    "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?",
    "What are the two RAG model variants proposed in the paper?",
    "How many attention heads does the base Transformer model use?",
]

print("=== B4: Español vs Inglés ===")
for q_es, q_en in zip(preguntas_es, preguntas_en_equiv):
    print(f"\n--- ES: {q_es} ---")
    resp_es = invoke_con_reintento(rag_chain, q_es)
    print(f"A: {resp_es}")
    docs_es = retriever.invoke(q_es)
    print(f"Chunks recuperados (ES): {[d.metadata.get('source') for d in docs_es]}")

    print(f"\n--- EN: {q_en} ---")
    resp_en = invoke_con_reintento(rag_chain, q_en)
    print(f"A: {resp_en}")
    docs_en = retriever.invoke(q_en)
    print(f"Chunks recuperados (EN): {[d.metadata.get('source') for d in docs_en]}")
    time.sleep(2)

=== B4: Español vs Inglés ===

--- ES: ¿Qué puntuación BLEU logró el Transformer (modelo grande) en WMT 2014 inglés-alemán? ---
A: El modelo Transformer (big) logró una puntuación BLEU de **28.4** en la tarea de traducción WMT 2014 inglés-alemán.

**Fuente:** *Attention Is All You Need*
Chunks recuperados (ES): ['attention_paper.pdf', 'attention_paper.pdf', 'attention_paper.pdf', 'attention_paper.pdf', 'attention_paper.pdf']

--- EN: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German? ---
A: The Transformer (big model) achieved a BLEU score of **28.4** on the WMT 2014 English-to-German translation task.

**Supporting paper:** *Attention Is All You Need*
Chunks recuperados (EN): ['attention_paper.pdf', 'attention_paper.pdf', 'attention_paper.pdf', 'attention_paper.pdf', 'attention_paper.pdf']

--- ES: ¿Cuáles son las dos variantes del modelo RAG propuestas en el paper? ---
A: Las dos variantes del modelo RAG propuestas en el paper son **RAG-Sequence** 

In [34]:
mi_quiz = [
    {
        "pregunta": "What dataset did the Transformer use for the English-to-French translation task?",
        "ground_truth": "WMT 2014 English-French dataset",
    },
    {
        "pregunta": "What retrieval method does RAG use to access the non-parametric memory?",
        "ground_truth": "A pre-trained neural retriever (DPR) over a dense vector index of Wikipedia",
    },
    {
        # esta debe combinar info de los dos papers
        "pregunta": "Does the RAG paper use the Transformer architecture proposed in Attention Is All You Need, and in which component?",
        "ground_truth": "Yes, indirectly — RAG's generator (BART) is a seq2seq Transformer model",
    },
]

print("=== B5: Mi propio quiz ===")
for item in mi_quiz:
    resp = invoke_con_reintento(rag_chain, item["pregunta"])
    print(f"\nQ: {item['pregunta']}\nGround truth: {item['ground_truth']}\nA: {resp}\n")
    time.sleep(2)

=== B5: Mi propio quiz ===

Q: What dataset did the Transformer use for the English-to-French translation task?
Ground truth: WMT 2014 English-French dataset
A: The Transformer used the **WMT 2014 English-French dataset** for the English-to-French translation task, which consists of **36 million sentence pairs** with tokens split into a 32,000 word-piece vocabulary.

**Paper:** Attention Is All You Need (attention_paper.pdf)


Q: What retrieval method does RAG use to access the non-parametric memory?
Ground truth: A pre-trained neural retriever (DPR) over a dense vector index of Wikipedia
A: RAG uses the **Dense Passage Retriever (DPR)**, a pre-trained neural retriever, to access a dense vector index of Wikipedia as its non-parametric memory. The retriever provides latent documents conditioned on the input, which the seq2seq generator (BART) then conditions on to generate the output.

**Paper title:** Retrieval-Augmented Generation


Q: Does the RAG paper use the Transformer architectu

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.
